# 06.7 — Mini-GPT: Character-Level Language Model

**The culmination of Modules 05-06.** Train a small Transformer on text and generate coherent continuations.

**Following Karpathy's nanoGPT:** Minimal, readable implementation. The full power of the architecture in ~200 lines.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ---- nanoGPT-style implementation ----

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)  # Q, K, V in one projection
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd
        self.register_buffer('bias', torch.tril(torch.ones(block_size, block_size))
                                          .view(1, 1, block_size, block_size))
    
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        
        att = (q @ k.transpose(-2,-1)) * (1.0/math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = att @ v
        y = y.transpose(1,2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc   = nn.Linear(n_embd, 4*n_embd)
        self.c_proj = nn.Linear(4*n_embd, n_embd)
        self.drop   = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.c_proj(F.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp  = MLP(n_embd, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size=128, n_embd=128, n_head=4, 
                 n_layer=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.transformer = nn.ModuleDict({
            'wte': nn.Embedding(vocab_size, n_embd),
            'wpe': nn.Embedding(block_size, n_embd),
            'drop': nn.Dropout(dropout),
            'h': nn.ModuleList([Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]),
            'ln_f': nn.LayerNorm(n_embd),
        })
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        # Weight tying: share embedding and output weights
        self.transformer.wte.weight = self.lm_head.weight
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)
                if isinstance(m, nn.Linear) and m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size
        
        pos = torch.arange(T, device=idx.device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, 1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Character-level tokenizer
text = """
The transformer is a deep learning model that adopts the mechanism of self-attention.
It was introduced in 2017 and has since become the foundation of modern NLP.
Models like BERT and GPT are built on the transformer architecture.
Attention is all you need was the title of the paper that introduced this architecture.
The model processes sequences of tokens and learns contextual representations.
Self-attention allows each token to attend to all other tokens in the sequence.
The feedforward network processes each token independently after the attention step.
Residual connections and layer normalization stabilize the training process.
Positional encodings inject information about the position of each token.
The decoder uses masked self-attention to prevent attending to future tokens.
""".strip() * 5  # repeat for more training data

chars = sorted(set(text))
vocab_size = len(chars)
c2i = {c:i for i,c in enumerate(chars)}
i2c = {i:c for c,i in c2i.items()}

data = torch.tensor([c2i[c] for c in text], dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]

print(f"Vocab size: {vocab_size}, Data length: {len(data)}")

# Mini config
BLOCK_SIZE = 64
model = MiniGPT(vocab_size=vocab_size, block_size=BLOCK_SIZE, 
                 n_embd=64, n_head=4, n_layer=3, dropout=0.1)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
import random

def get_batch(data, block_size, batch_size=32):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

for step in range(2000):
    xb, yb = get_batch(train_data, BLOCK_SIZE)
    _, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    if step % 500 == 0:
        model.eval()
        with torch.no_grad():
            xv, yv = get_batch(val_data, BLOCK_SIZE, batch_size=64)
            _, val_loss = model(xv, yv)
        model.train()
        print(f"Step {step:4d}: train_loss={loss.item():.4f}  val_loss={val_loss.item():.4f}")

# Generate text
print("\n=== Generated Text ===")
for temp in [0.5, 1.0, 1.5]:
    seed = torch.tensor([[c2i['T']]])
    generated = model.generate(seed, max_new_tokens=100, temperature=temp, top_k=5)
    text_out = ''.join(i2c[i] for i in generated[0].tolist())
    print(f"\nTemperature={temp}:")
    print(f"  {text_out!r}")

## Summary — Module 06 Complete

You've built the full Transformer from scratch:
1. Bahdanau attention → intuition
2. Scaled dot-product attention → the core operation
3. Multi-head attention → parallel representation learning
4. Positional encoding → injecting order
5. Transformer encoder → contextual representations
6. Transformer decoder → causal language modeling
7. Mini-GPT → it all comes together

**What next-token prediction learns:** The model must predict the next character/token at every position. To do this well, it must learn grammar, vocabulary, context, and factual associations. This is why self-supervised pretraining on text is so powerful.

---
**Module 07: Modern NLP — BERT, fine-tuning, RAG, LoRA**